In [ ]:
import os
import chromadb
from pypdf import PdfReader
from google import genai

ai_client = genai.Client(api_key="YOUR_GEMINI_API_KEY")

CHUNK_SIZE = 600
filepath = "document.pdf"

# Initialize vector db
db_client = chromadb.Client()
collection = db_client.create_collection(name="pdf_data")

In [ ]:
# Parsing
reader = PdfReader(filepath)
text = ""
for page in reader.pages:
    text += page.extract_text()

# Split into chunks based on size
chunks = [text[i:i+CHUNK_SIZE] for i in range(0, len(text), CHUNK_SIZE)]

# Add to vector db
for i, chunk in enumerate(chunks):
    response = ai_client.models.embed_content(
        model="gemini-embedding-001",
        contents=chunk
    )
    vec = response.embeddings[0].values
    collection.add(ids=[str(i)], embeddings=[vec], documents=[chunk])

print("Data loaded.")

In [ ]:
def ask_question(question):
    # Vectorize query
    q_vec = ai_client.models.embed_content(
        model="gemini-embedding-001",
        contents=question
    ).embeddings[0].values
    
    # Retrieve context
    results = collection.query(query_embeddings=[q_vec], n_results=2)
    context = results['documents'][0][0]
    
    # Generate answer
    prompt = f"Context: {context}\n\nQuestion: {question}"
    response = ai_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text

# Simple interaction
while True:
    q = input("Question: ")
    if q == "quit": break
    print(ask_question(q))

In [ ]:
def generate_rag_response(user_input: str) -> str:
    """Retrieves context and asks the LLM, heavily guarded with try-except blocks."""
    try:
        input_vector = fetch_vector(user_input)
        if not input_vector:
            return "System Error: Could not vectorize your query."

        # Changed n_results to 4 to give the model slightly more context
        search_results = knowledge_base.query(
            query_embeddings=[input_vector],
            n_results=4  
        )
        
        found_docs = search_results.get("documents", [[]])[0]
        context_block = "\n\n---\n\n".join(found_docs)
        
        # Reformatted prompt structure
        strict_prompt = (
            f"You are an AI assistant. Answer the user's question STRICTLY based on the provided context.\n\n"
            f"CONTEXT:\n{context_block}\n\n"
            f"USER QUESTION: {user_input}"
        )
        
        reply = ai_client.models.generate_content(
            model=CHAT_MODEL,
            contents=strict_prompt,
        )
        return reply.text
        
    except Exception as api_err:
        return f"Generation Error: {api_err}"

# Execution loop
print("--- RAG Assistant Online ---")
while True:
    question = input("Ask a question (or type 'quit'): ").strip()
    
    if question.lower() in ['quit', 'exit']:
        print("Shutting down...")
        break
    if not question:
        continue
        
    print("\nThinking...")
    answer = generate_rag_response(question)
    print(f"\nAI: {answer}\n")
    print("-" * 50)